# Main Notebook (Current API)

This notebook recreates the main themes of the original `main.ipynb`, but uses the current `mykalshi` codebase.

It focuses on:

- live market discovery and overview
- event and market drilldown
- current order book visualization
- session-based websocket capture
- replay/backtest on captured data
- climate and weather universe exploration

It is read-only with respect to the exchange. The only file-writing step is a local temporary replay session directory.

In [ ]:
import importlib.util
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

missing = [
    package_name
    for package_name in ("matplotlib", "pandas", "seaborn")
    if importlib.util.find_spec(package_name) is None
]
if missing:
    install_target = f'{PROJECT_ROOT}[analysis,storage,websocket]'
    raise ModuleNotFoundError(
        "Notebook dependencies are missing in the current kernel "
        f'({sys.executable}). Missing: {", ".join(missing)}. '
        f'Run `%pip install -e \"{install_target}\"` in this notebook, '
        'or switch VS Code/Jupyter to the "mykalshi (.venv)" kernel.'
    )

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from mykalshi import events, market
from mykalshi.orderbook import extract_orderbook_levels
from mykalshi.research import KalshiStrategy, ResearchSession

sns.set_theme(style="darkgrid")
%config InlineBackend.figure_format = "retina"

session = ResearchSession()


def choose_live_event_and_market(search_limit=25):
    event_rows = events.get_events(status="open", limit=search_limit)["events"]
    for event_row in event_rows:
        market_rows = market.get_markets(
            event_ticker=event_row["event_ticker"],
            status="open",
            limit=1,
        )["markets"]
        if market_rows:
            return event_row, market_rows[0]
    raise RuntimeError("Could not find an open event with at least one open market.")


def orderbook_to_yes_frames(orderbook_payload):
    yes_levels, no_levels = extract_orderbook_levels(orderbook_payload)
    yes_bids = pd.DataFrame(
        [
            {"price_cents": price_cents, "size": float(size)}
            for price_cents, size in sorted(yes_levels.items(), reverse=True)
        ]
    )
    yes_asks = pd.DataFrame(
        [
            {"price_cents": 100 - price_cents, "size": float(size)}
            for price_cents, size in sorted(no_levels.items(), reverse=True)
        ]
    )
    if not yes_asks.empty:
        yes_asks = yes_asks.sort_values("price_cents")
    return yes_bids, yes_asks


def plot_yes_orderbook(orderbook_payload, title):
    yes_bids, yes_asks = orderbook_to_yes_frames(orderbook_payload)
    fig, ax = plt.subplots(figsize=(10, 5))
    if not yes_bids.empty:
        bid_curve = yes_bids.sort_values("price_cents")
        ax.step(
            bid_curve["price_cents"],
            bid_curve["size"].cumsum(),
            where="post",
            label="YES bids",
            color="green",
        )
    if not yes_asks.empty:
        ask_curve = yes_asks.sort_values("price_cents")
        ax.step(
            ask_curve["price_cents"],
            ask_curve["size"].cumsum(),
            where="post",
            label="YES asks",
            color="red",
        )
    ax.set_title(title)
    ax.set_xlabel("Price (cents)")
    ax.set_ylabel("Cumulative size")
    ax.set_xlim(0, 100)
    ax.legend()
    plt.tight_layout()
    plt.show()


def candlesticks_response_to_frame(response):
    rows = []
    for candle in response.get("candlesticks", []):
        row = {
            "end_period": pd.to_datetime(candle["end_period_ts"], unit="s", utc=True),
            "volume_fp": float(candle.get("volume_fp", 0) or 0),
            "open_interest_fp": float(candle.get("open_interest_fp", 0) or 0),
        }
        for section in ("price", "yes_bid", "yes_ask"):
            for key, value in candle.get(section, {}).items():
                row[f"{section}_{key}"] = float(value) if value is not None else None
        rows.append(row)
    return pd.DataFrame(rows)


class BuyFirstOrderbookStrategy(KalshiStrategy):
    def __init__(self):
        self.submitted = False

    def on_orderbook(self, context, event):
        if self.submitted:
            return
        context.buy_yes(event.market_ticker, quantity=1)
        self.submitted = True


## 1. Open Market Overview

The original notebook started with broad market analysis. Here we sample the current live open markets with the modern wrapper layer.

In [ ]:
open_markets = pd.json_normalize(
    market.get_all_markets(status="open", batch_size=100, max_items=200)
)
volume_series = open_markets.get("volume_fp", open_markets.get("volume_24h_fp"))
open_markets["volume_proxy"] = pd.to_numeric(volume_series, errors="coerce").fillna(0)

top_open_markets = (
    open_markets[["ticker", "title", "status", "volume_proxy"]]
    .sort_values("volume_proxy", ascending=False)
    .head(15)
)
display(top_open_markets)

ax = (
    top_open_markets.sort_values("volume_proxy")
    .plot.barh(x="ticker", y="volume_proxy", figsize=(12, 6), legend=False)
)
ax.set_title("Top open markets by volume_fp (sample of 200)")
ax.set_xlabel("Volume_fp")
ax.set_ylabel("Market ticker")
plt.tight_layout()


## 2. Event Drilldown And Candlesticks

The original notebook drilled into a specific event and its markets. Here we resolve one live event dynamically so the notebook stays usable over time.

In [ ]:
live_event, live_market = choose_live_event_and_market()
live_market_meta = market.get_market(live_market["ticker"])["market"]
event_snapshot = events.event_info(live_event["event_ticker"])

selection = pd.DataFrame(
    [
        {
            "series_ticker": live_event["series_ticker"],
            "event_ticker": live_event["event_ticker"],
            "event_title": live_event["title"],
            "market_ticker": live_market["ticker"],
            "market_title": live_market["title"],
            "market_status": live_market["status"],
            "market_close_time": live_market_meta["close_time"],
        }
    ]
)
display(selection)
display(event_snapshot["markets"].head(10))

open_time = pd.Timestamp(live_market_meta["open_time"], tz="UTC")
close_time = pd.Timestamp(live_market_meta["close_time"], tz="UTC")
end_time = min(pd.Timestamp.now(tz="UTC"), close_time)
start_time = max(open_time, end_time - pd.Timedelta(days=2))

candles = market.get_market_candlesticks(
    series_ticker=live_event["series_ticker"],
    ticker=live_market["ticker"],
    start_ts=start_time.tz_convert(None).strftime("%m/%d/%Y %H:%M:%S"),
    end_ts=end_time.tz_convert(None).strftime("%m/%d/%Y %H:%M:%S"),
    period_interval=60,
)
candles_df = candlesticks_response_to_frame(candles)
if not candles_df.empty:
    candles_df = candles_df.sort_values("end_period")
display(candles_df.tail())

price_column = next(
    (column for column in ("price_close_dollars", "yes_bid_close_dollars", "yes_ask_close_dollars") if column in candles_df.columns),
    None,
)
if price_column is not None and not candles_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(candles_df["end_period"], candles_df[price_column], label=price_column)
    ax.set_title(f"Recent candlestick history for {live_market['ticker']}")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Price")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 3. Order Book Tools

The old notebook used the legacy REST order book shape. The current code supports Kalshi's newer fixed-point `orderbook_fp` responses, so the notebook now normalizes the live book through `extract_orderbook_levels(...)`.

In [ ]:
live_orderbook = market.get_market_orderbook(live_market["ticker"])
yes_bids, yes_asks = orderbook_to_yes_frames(live_orderbook)

display(yes_bids.head(10))
display(yes_asks.head(10))
plot_yes_orderbook(live_orderbook, f"YES depth for {live_market['ticker']}")


## 4. Session Capture, Replay, And Backtest

This is the modern research workflow that did not exist in the original notebook: capture a standardized replay session, load it back, and run a replay backtest on the captured market data.

In [ ]:
session_dir = Path(tempfile.gettempdir()) / "mykalshi-main-current-session"
capture = session.capture_market_session(
    session_dir,
    market_ticker=live_market["ticker"],
    channels=["orderbook_delta"],
    max_events=1,
    overwrite=True,
)
display(pd.DataFrame([capture.summary()]))

replay_dataset = session.load_replay_dataset(session_dir=session_dir)
display(pd.DataFrame([replay_dataset.summary()]))

replay_result = session.run_replay_backtest(
    BuyFirstOrderbookStrategy(),
    session_dir=session_dir,
    enrich_market_lifecycle=False,
    initial_cash_cents=200,
)
display(pd.DataFrame([replay_result.summary()]))

replay_frames = replay_result.to_dataframes()
display(replay_frames["fills"])


## 5. Climate And Weather Universe Snapshot

The original notebook spent a lot of time in weather/event-family exploration. The current discovery layer makes that much easier.

In [ ]:
weather_series = pd.DataFrame(
    [item.summary() for item in session.search_series(category="Climate and Weather", limit=25)]
)
weather_events = pd.DataFrame(
    [item.summary() for item in session.search_events(category="Climate and Weather", status="open", limit=25)]
)
weather_markets = pd.DataFrame(
    [item.summary() for item in session.search_markets(category="Climate and Weather", status="active", limit=25)]
)

display(weather_series.head(10))
display(weather_events.head(10))
display(weather_markets.head(10))

if not weather_markets.empty and "series_ticker" in weather_markets.columns:
    counts = weather_markets["series_ticker"].value_counts().head(10)
    ax = counts.sort_values().plot.barh(figsize=(10, 5))
    ax.set_title("Climate and Weather market counts by series (sample)")
    ax.set_xlabel("Market count")
    ax.set_ylabel("Series ticker")
    plt.tight_layout()
